# 00 — Map External Data Sources to Pipeline Format

Converts external AML datasets into the **three CSV files** expected by
notebook 01 (`../demodata/`):

| File | Columns |
|------|----------|
| `party.csv` | `partyId, partyType` |
| `transactions.csv` | `tran_id, tx_type, base_amt, tran_timestamp, src, dst` |
| `alert_transactions.csv` | `alert_id, alert_type, is_sar, tran_id` |

### Supported Sources
- **SAML-D** — IBM synthetic AML dataset (`SAML-D.csv`)
- *(Add new mappers below for future datasets)*

### tx_type Convention
Notebook 01 encodes via `split('-')[0]`:
`CASH_IN=0, CASH_OUT=1, DEBIT=2, PAYMENT=3, TRANSFER/DEPOSIT=4`

So `TRANSFER-ACH` → `TRANSFER` → 4.  Use this prefix convention for new sources.

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import os

# ═══════════════════════════════════════════════════════════
#  CONFIGURATION — Change these for your data source
# ═══════════════════════════════════════════════════════════
DATA_SOURCE = "SAML-D"          # "SAML-D" | "custom" (add your own mapper)
INPUT_PATH  = "/mnt/e/xx/SAML-D.csv"   # Path to the raw dataset
OUTPUT_DIR  = os.path.join("..", "demodata")  # Pipeline expects ../demodata/

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Source:  {DATA_SOURCE}")
print(f"Input:   {INPUT_PATH}")
print(f"Output:  {os.path.abspath(OUTPUT_DIR)}")

---
## SAML-D Mapper

IBM's Synthetic AML Dataset. Columns include:
`Date, Time, Sender_account, Receiver_account, Amount, Payment_currency,
Received_currency, Sender_bank_location, Receiver_bank_location,
Payment_type, Is_laundering, Laundering_type`

In [ ]:
def map_samld(input_path):
    """Map SAML-D.csv → (party_df, transactions_df, alerts_df, extra_df)"""
    df = pd.read_csv(input_path)
    print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
    print(f"Columns: {list(df.columns)}")
    print(f"Laundering rate: {df['Is_laundering'].mean():.4%}")
    print(f"Unique senders: {df['Sender_account'].nunique():,}")
    print(f"Unique receivers: {df['Receiver_account'].nunique():,}")

    # ── partyId: 8-char hex hash of account number ──
    all_accounts = pd.concat([df["Sender_account"], df["Receiver_account"]]).unique()
    account_to_pid = {
        acct: hashlib.md5(str(acct).encode()).hexdigest()[:8]
        for acct in all_accounts
    }
    unique_hashes = len(set(account_to_pid.values()))
    if unique_hashes < len(all_accounts):
        print(f"WARNING: {len(all_accounts) - unique_hashes} hash collisions!")
    else:
        print(f"No hash collisions ({len(all_accounts):,} accounts → {unique_hashes:,} hashes)")

    df["src"] = df["Sender_account"].map(account_to_pid)
    df["dst"] = df["Receiver_account"].map(account_to_pid)

    # ── tran_timestamp: ISO 8601 ──
    df["tran_timestamp"] = pd.to_datetime(
        df["Date"] + "T" + df["Time"]
    ).dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")

    # ── tran_id: sequential ──
    df["tran_id"] = range(1, len(df) + 1)

    # ── tx_type: map to pipeline prefix convention ──
    PAYMENT_TYPE_MAP = {
        "Cash Deposit":    "CASH_IN",
        "Cash Withdrawal": "CASH_OUT",
        "ACH":             "TRANSFER-ACH",
        "Cheque":          "PAYMENT-Cheque",
        "Credit card":     "PAYMENT-CreditCard",
        "Debit card":      "DEBIT-Card",
        "Cross-border":    "TRANSFER-CrossBorder",
    }
    df["tx_type"] = df["Payment_type"].map(PAYMENT_TYPE_MAP)
    unmapped = df["tx_type"].isna().sum()
    if unmapped > 0:
        print(f"WARNING: {unmapped} rows with unmapped Payment_type:")
        print(df.loc[df["tx_type"].isna(), "Payment_type"].unique())
        df["tx_type"] = df["tx_type"].fillna("TRANSFER")  # fallback

    # ── alert_type: map laundering types ──
    LAUNDERING_TYPE_MAP = {
        "Cycle":               "cycle",
        "Gather-Scatter":      "gather_scatter",
        "Scatter-Gather":      "scatter_gather",
        "Fan_In":              "fan_in",
        "Fan_Out":             "fan_out",
        "Layered_Fan_In":      "fan_in",
        "Layered_Fan_Out":     "fan_out",
        "Smurfing":            "structuring",
        "Structuring":         "structuring",
        "Bipartite":           "bipartite",
        "Stacked Bipartite":   "bipartite",
        "Deposit-Send":        "deposit_send",
        "Cash_Withdrawal":     "cash_withdrawal",
        "Over-Invoicing":      "over_invoicing",
        "Single_large":        "single_large",
        "Behavioural_Change_1": "behavioural_change",
        "Behavioural_Change_2": "behavioural_change",
    }
    df["alert_type"] = df["Laundering_type"].map(LAUNDERING_TYPE_MAP)
    df.loc[df["Is_laundering"] == 0, "alert_type"] = None

    unmapped_launder = df[(df["Is_laundering"] == 1) & (df["alert_type"].isna())]
    if len(unmapped_launder) > 0:
        print(f"WARNING: {len(unmapped_launder)} laundering rows unmapped:")
        print(unmapped_launder["Laundering_type"].unique())

    # ── Build party.csv ──
    party_ids = sorted(set(df["src"].unique()) | set(df["dst"].unique()))
    party_df = pd.DataFrame({"partyId": party_ids, "partyType": "Individual"})

    # ── Build transactions.csv ──
    transactions_df = df[["tran_id", "tx_type", "Amount", "tran_timestamp", "src", "dst"]].copy()
    transactions_df = transactions_df.rename(columns={"Amount": "base_amt"})

    # ── Build alert_transactions.csv ──
    launder_df = df[df["Is_laundering"] == 1].copy()
    alert_type_to_id = {
        atype: idx + 1
        for idx, atype in enumerate(sorted(launder_df["alert_type"].unique()))
    }
    launder_df["alert_id"] = launder_df["alert_type"].map(alert_type_to_id)
    alerts_df = launder_df[["alert_id", "alert_type", "tran_id"]].copy()
    alerts_df["is_sar"] = "true"
    alerts_df = alerts_df[["alert_id", "alert_type", "is_sar", "tran_id"]]

    # ── Extra reference info (preserved for analysis) ──
    extra_df = df[[
        "tran_id", "Sender_account", "Receiver_account",
        "Payment_currency", "Received_currency",
        "Sender_bank_location", "Receiver_bank_location",
        "Payment_type", "Laundering_type", "Is_laundering"
    ]].copy()

    return party_df, transactions_df, alerts_df, extra_df

---
## Custom Mapper Template

Uncomment and adapt this for a new data source. Your mapper must return
the same 4-tuple: `(party_df, transactions_df, alerts_df, extra_df)`.

In [ ]:
# def map_custom(input_path):
#     """Template mapper for a new AML data source."""
#     df = pd.read_csv(input_path)
#
#     # ── party.csv ──
#     # Must have: partyId (str), partyType ("Individual" or "Organization")
#     party_df = pd.DataFrame({
#         "partyId": [...],
#         "partyType": [...],
#     })
#
#     # ── transactions.csv ──
#     # Must have: tran_id (int), tx_type (str, prefix convention), base_amt (float),
#     #            tran_timestamp (ISO 8601 str), src (partyId), dst (partyId)
#     # tx_type prefix: CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER, DEPOSIT
#     transactions_df = pd.DataFrame({
#         "tran_id": [...],
#         "tx_type": [...],
#         "base_amt": [...],
#         "tran_timestamp": [...],
#         "src": [...],
#         "dst": [...],
#     })
#
#     # ── alert_transactions.csv ──
#     # Must have: alert_id (int), alert_type (str), is_sar ("true"), tran_id (int)
#     alerts_df = pd.DataFrame({
#         "alert_id": [...],
#         "alert_type": [...],
#         "is_sar": "true",
#         "tran_id": [...],
#     })
#
#     extra_df = pd.DataFrame()  # optional reference data
#
#     return party_df, transactions_df, alerts_df, extra_df

---
## Run Selected Mapper

In [ ]:
MAPPERS = {
    "SAML-D": map_samld,
    # "custom": map_custom,
}

if DATA_SOURCE not in MAPPERS:
    raise ValueError(f"Unknown DATA_SOURCE='{DATA_SOURCE}'. Available: {list(MAPPERS.keys())}")

party_df, transactions_df, alerts_df, extra_df = MAPPERS[DATA_SOURCE](INPUT_PATH)

print(f"\n{'='*50}")
print(f"  party.csv:              {len(party_df):>10,} rows")
print(f"  transactions.csv:       {len(transactions_df):>10,} rows")
print(f"  alert_transactions.csv: {len(alerts_df):>10,} rows")
if extra_df is not None and len(extra_df) > 0:
    print(f"  extra_info.csv:         {len(extra_df):>10,} rows")
print(f"{'='*50}")

## Data Overview

In [ ]:
print("=== tx_type distribution ===")
print(transactions_df["tx_type"].value_counts())

print("\n=== alert_type distribution ===")
print(alerts_df["alert_type"].value_counts())

print("\n=== partyType distribution ===")
print(party_df["partyType"].value_counts())

print("\n=== base_amt stats ===")
print(transactions_df["base_amt"].describe())

## Save Output

In [ ]:
party_df.to_csv(os.path.join(OUTPUT_DIR, "party.csv"), index=False)
transactions_df.to_csv(os.path.join(OUTPUT_DIR, "transactions.csv"), index=False)
alerts_df.to_csv(os.path.join(OUTPUT_DIR, "alert_transactions.csv"), index=False)
if extra_df is not None and len(extra_df) > 0:
    extra_df.to_csv(os.path.join(OUTPUT_DIR, "extra_info.csv"), index=False)

print("=== Files saved ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {f:40s} {size_mb:.1f} MB")

## Validation

Cross-check that the output matches what notebook 01 expects.

In [ ]:
errors = []

# ── party.csv checks ──
p = pd.read_csv(os.path.join(OUTPUT_DIR, "party.csv"))
for col in ["partyId", "partyType"]:
    if col not in p.columns:
        errors.append(f"party.csv missing column: {col}")
print(f"party.csv: {len(p):,} rows, cols={list(p.columns)}")

# ── transactions.csv checks ──
t = pd.read_csv(os.path.join(OUTPUT_DIR, "transactions.csv"))
for col in ["tran_id", "tx_type", "base_amt", "tran_timestamp", "src", "dst"]:
    if col not in t.columns:
        errors.append(f"transactions.csv missing column: {col}")
print(f"transactions.csv: {len(t):,} rows, cols={list(t.columns)}")

# Verify tx_type prefixes are recognized
VALID_PREFIXES = {"CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER", "DEPOSIT"}
prefixes = t["tx_type"].apply(lambda x: str(x).split("-")[0]).unique()
unknown = set(prefixes) - VALID_PREFIXES
if unknown:
    errors.append(f"Unknown tx_type prefixes (will map to code 99): {unknown}")
print(f"tx_type prefixes: {sorted(prefixes)}")

# ── alert_transactions.csv checks ──
a = pd.read_csv(os.path.join(OUTPUT_DIR, "alert_transactions.csv"))
for col in ["alert_id", "alert_type", "is_sar", "tran_id"]:
    if col not in a.columns:
        errors.append(f"alert_transactions.csv missing column: {col}")
print(f"alert_transactions.csv: {len(a):,} rows, cols={list(a.columns)}")

# ── Cross-referential integrity ──
all_party_ids = set(p["partyId"])
missing_src = set(t["src"]) - all_party_ids
missing_dst = set(t["dst"]) - all_party_ids
missing_tran = set(a["tran_id"]) - set(t["tran_id"])

if missing_src:
    errors.append(f"{len(missing_src)} src IDs not in party.csv")
if missing_dst:
    errors.append(f"{len(missing_dst)} dst IDs not in party.csv")
if missing_tran:
    errors.append(f"{len(missing_tran)} alert tran_ids not in transactions.csv")

print(f"\nReferential integrity: src={len(missing_src)}, dst={len(missing_dst)}, tran={len(missing_tran)} missing")

if errors:
    print(f"\n{'!'*50}")
    print(f"  {len(errors)} VALIDATION ERROR(S):")
    for e in errors:
        print(f"  - {e}")
    print(f"{'!'*50}")
else:
    print(f"\n{'='*50}")
    print(f"  ALL VALIDATIONS PASSED")
    print(f"  Ready for: e2e/01_data_loading_and_features.ipynb")
    print(f"{'='*50}")